# 10 · Expert behaviour by phase

**What this shows.** How the animals behave once trained, per stimulus distribution (Uniform, Hard-A,
Hard-B): the psychometric curve and its parameters, serial dependence, the update matrix — first for one
animal, then across the cohort split by genotype. No opto in this notebook: everything here is the
laser-off side of the opto sessions, i.e. ordinary expert behaviour.

**What produced it.** An `opto_contrasts` run of `sc-reports`. Every per-animal folder of that run holds
`contrasts.csv` (the contrasts, notebook 20), `readouts.npz` (curves and update matrices per condition
and trial type — only when the run was not `--fast`), and the per-session `trajectory.csv`.

```bash
sc-reports opto --all --cohort opto1-cohort        # every distribution × design × toi (hours)
sc-reports opto --distribution Uniform              # one condition
```

Tables, not recomputation: this notebook reads the run `latest` points at (`SC_NB_RUN` to pin one).

In [ ]:
from nb_setup import *
banner()
run = open_run('opto_contrasts', OPTO_COHORT)
contrasts = load_contrasts(run)
rows, tests = load_group(run)
print(sorted(contrasts.distribution.unique()), '| animals:', sorted(contrasts.animal.unique()))

## 1 · One animal, one phase: the pipeline in five lines

The reports compute everything through the library pipeline `load → select_sessions → filter_trials →
pool_arrays → compute_x → plot_x`. Here it is once, by hand, so the numbers in the tables are traceable.

In [ ]:
from behav_utils import (PSYCHOMETRIC, TrialArrays, compute_psychometric_curve, compute_stats, compute_update_matrix,
                         filter_trials, plot_psychometric_curve, plot_update_matrix, select_sessions)

exp = load_experiment()
aid = first_animal(contrasts)
animal = exp.animals[aid]
dist = 'Uniform'

sessions = select_sessions(animal, distribution=dist, session_type='opto')     # the opto sessions ...
clean = filter_trials(sessions, trial_type='non_opto')                         # ... laser-off trials only
arrays = TrialArrays.from_sessions(clean)
stats = compute_stats(arrays, ['accuracy', 'side_bias', 'recency', *PSYCHOMETRIC])
curve = compute_psychometric_curve(arrays, n_bootstrap=100)
um = compute_update_matrix(arrays)

fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
plot_psychometric_curve(curve, ax=axes[0], color=PALETTE[0])
plot_update_matrix(um, ax=axes[1])
axes[0].set_title(f'{aid} {dist}, laser-off ({arrays.n_trials} trials)')
stats.round(3)

## 2 · Every animal, every phase: the laser-off statistics

The group fold writes one value per animal and stat into `group_rows.csv`. The `kind == 'within'` rows
are differences; the per-condition *levels* live in `contrasts.csv` as the `non_opto` phase summaries.
Here we read the laser-off levels straight from each animal's `readouts.npz` curve parameters when the
run has them, and fall back to the contrast tables otherwise.

In [ ]:
def laser_off_params(run, animals, distribution):
    out = []
    for aid in animals:
        r = load_readouts(run, aid, distribution)
        key = 'opto__non_opto__curve_params'
        if key in r:
            mu, sigma, ll, lh = r[key]
            out.append({'animal': aid, 'distribution': distribution, 'mu': mu, 'sigma': sigma,
                        'lapse_low': ll, 'lapse_high': lh})
    return pd.DataFrame(out)

genotype = contrasts.drop_duplicates('animal').set_index('animal')['genotype']
levels = pd.concat([laser_off_params(run, sorted(contrasts.animal.unique()), d)
                    for d in sorted(contrasts.distribution.unique())], ignore_index=True)
if len(levels):
    levels['genotype'] = levels.animal.map(genotype)
    display(levels.pivot_table(index=['genotype', 'animal'], columns='distribution', values='mu').round(3))
else:
    print('this run was --fast: no readouts.npz; rerun without --fast for per-animal curve parameters')

In [ ]:
# laser-off psychometric curves across animals, one panel per distribution
dists = sorted(contrasts.distribution.unique())
fig, axes = plt.subplots(1, len(dists), figsize=(4 * len(dists), 3.4), sharey=True)
drawn = 0
for ax, d in zip(np.atleast_1d(axes), dists):
    for aid in sorted(contrasts.animal.unique()):
        r = load_readouts(run, aid, d)
        if 'opto__non_opto__curve_x' in r:
            ax.plot(r['opto__non_opto__curve_x'], r['opto__non_opto__curve_y'],
                    color=GENOTYPE_COLOUR.get(genotype[aid], '0.5'), alpha=0.6, lw=1)
            drawn += 1
    ax.axhline(0.5, color='0.8', lw=0.6); ax.axvline(0, color='0.8', lw=0.6)
    ax.set(title=f'{d}, laser-off', xlabel='stimulus', ylim=(0, 1))
axes[0].set_ylabel('P(choose B)') if hasattr(axes, '__len__') else None
plt.tight_layout()
if not drawn:
    print('no readouts in this run (--fast); nothing drawn')

## 3 · Update matrices by phase

The update matrix (the shift in P(B | current bin) by the previous stimulus bin, after a correct trial) is the readout that separates
the two candidate models (notebook 31): a boundary-estimation animal shows a near-uniform matrix, a
statistical-categorisation animal a structured one. Mean laser-off matrix per genotype and distribution.

In [ ]:
fig, axes = plt.subplots(2, len(dists), figsize=(3.2 * len(dists), 6), squeeze=False)
for j, d in enumerate(dists):
    for i, g in enumerate(('wt', 'het')):
        mats = [load_readouts(run, aid, d).get('opto__non_opto__um') for aid in genotype[genotype == g].index]
        mats = [m for m in mats if m is not None]
        ax = axes[i, j]
        if mats:
            ax.imshow(np.nanmean(mats, axis=0), vmin=-0.4, vmax=0.4, cmap='RdBu_r', origin='lower')
            ax.set(title=f'{g.upper()} {d} (n={len(mats)})')
        else:
            ax.set(title=f'{g.upper()} {d}: no readouts'); ax.axis('off')
        ax.set(xlabel='previous stimulus bin', ylabel='current stimulus bin')
plt.tight_layout()

## 4 · Per-session trajectory

`trajectory.csv` has one row per session in order with the psychometric parameters, accuracy and history
stats per session. Used properly in notebook 21 (the Hard-A/B adaptation question); here just the raw
material: PSE per session for every animal, coloured by distribution.

In [ ]:
traj = load_trajectory(run)
if len(traj):
    fig, ax = plt.subplots(figsize=(10, 3.2))
    for aid, sub in traj.sort_values(['animal', 'session_idx']).groupby('animal'):
        ax.plot(sub['session_idx'], sub['pse'], color=GENOTYPE_COLOUR.get(genotype.get(aid, 'unknown'), '0.5'),
                alpha=0.5, lw=0.8)
    ax.axhline(0, color='0.7', lw=0.6)
    ax.set(xlabel='session', ylabel='PSE', title='PSE per session, all animals (WT blue, HET orange)')
else:
    print('no trajectory tables in this run')